# Verify Bybit Spot Margin on `bybit_2` (S-047 T0)

**S-047 Tier-1 operator notebook — read-only diagnostic.**

## What this notebook does

Confirms that **Bybit Spot Margin Trading** has been enabled on the
`bybit_2` account (Account → Margin Mode → Enable Spot Margin) and
captures the live BTC borrow tier so the operator can paste the number
into the T1 PR's `accounts.yaml` `margin:` block.

It is the gating check before S-047 T1 starts:

> *"Bybit Spot Margin Trading must be enabled on the `bybit_2` account
> via the Bybit web UI. Until that toggle is on, every `isLeverage=1`
> order returns retCode 110007 (MARGIN_TRADING_NOT_ENABLED)."*
> — `docs/sprint-plans/S-047-bybit2-spot-margin.md` § 2.

## What this notebook does NOT do

- It does **not** flip Spot Margin on or off. That toggle is web-UI-only
  on Bybit, and even if the API supported it, flipping margin mode from
  a notebook would be a Tier-3 change (alters the risk profile of a
  live account). T0 is read-only verification.
- It does not place, cancel, or modify any exchange order.
- It does not edit `.env`, `accounts.yaml`, or any code.

## ▶ Open in Colab

**On this branch (during T0 review):**
https://colab.research.google.com/github/the-lizardking/ict-trading-bot/blob/claude/S-047-T0-margin-enable-notebook-xBvbM/notebooks/operator/enable_bybit_spot_margin.ipynb

**On `main` (after T0 merges):**
https://colab.research.google.com/github/the-lizardking/ict-trading-bot/blob/main/notebooks/operator/enable_bybit_spot_margin.ipynb

## Required Colab Secrets

| Name | What it holds |
|---|---|
| `VM_SSH_HOST` | VM hostname or public IP |
| `VM_SSH_USER` | SSH user on the VM (usually `ubuntu`) |

## Order of operations

1. Operator opens the Bybit web UI on the `bybit_2` account →
   Account → Margin Mode → **Enable Spot Margin**.
2. Operator runs **Cell 1** (mounts Drive + opens SSH).
3. Operator runs **Cell 2** (read-only diagnostic).
4. Read **Cell 3** (decision matrix) and act on what Cell 2 printed.
5. Read **Cell 4** (handoff to T1) once Cell 2 reports
   `spotMarginEnabled = True` and the max-borrow-BTC tier is captured.


In [ ]:
# Cell 1: SSH setup. Mirrors notebooks/operator/sweep_unlinked_packages.ipynb.
import os, shutil, stat, subprocess, tempfile
from google.colab import drive, userdata

print("⏳ Mounting Google Drive…")
try:
    drive.mount("/content/drive")
except Exception as exc:
    print(f"⚠️  drive.mount() raised: {exc}")

DEFAULT_SSH_KEY_NAMES = [
    "ict-bot-ovm-private.key",
    "vm_ssh_key", "id_rsa", "id_ed25519", "id_ecdsa",
]
DRIVE_FOLDER = "/content/drive/MyDrive/ICT_Bot_Secrets"

ssh_key_path = None
if os.path.exists("/content/drive/MyDrive"):
    for name in DEFAULT_SSH_KEY_NAMES:
        path = os.path.join(DRIVE_FOLDER, name)
        if os.path.exists(path):
            ssh_key_path = path
            break
if ssh_key_path is None:
    raise SystemExit(
        f"No SSH key found in {DRIVE_FOLDER}/. Place "
        "`ict-bot-ovm-private.key` there and re-run."
    )
print(f"✅ SSH key located: {ssh_key_path}")

host = userdata.get("VM_SSH_HOST")
user = userdata.get("VM_SSH_USER")
if not host or not user:
    raise SystemExit("VM_SSH_HOST + VM_SSH_USER must be set in Colab Secrets.")

tmpdir = tempfile.mkdtemp(prefix="enable-spot-margin-")
key_path = os.path.join(tmpdir, "vm_key")
shutil.copy(ssh_key_path, key_path)
os.chmod(key_path, stat.S_IRUSR | stat.S_IWUSR)

ssh_target = f"{user}@{host}"
ssh_opts = [
    "-i", key_path,
    "-o", "StrictHostKeyChecking=accept-new",
    "-o", "UserKnownHostsFile=" + os.path.join(tmpdir, "known_hosts"),
    "-o", "ConnectTimeout=15",
    "-o", "BatchMode=yes",
]

def run_ssh(cmd, *, label, allow_fail=False):
    proc = subprocess.run(
        ["ssh"] + ssh_opts + [ssh_target, cmd],
        capture_output=True, text=True,
    )
    if proc.returncode != 0 and not allow_fail:
        print(f"❌ {label} failed (exit {proc.returncode})")
        print((proc.stderr or proc.stdout)[:500])
        raise SystemExit(f"{label} failed")
    return (proc.stdout or "").strip()

print(f"⏳ Connecting to {ssh_target} …")
run_ssh("echo connected", label="SSH connectivity")
print("✅ SSH OK")

REPO_DIR = f"/home/{user}/ict-trading-bot"


In [ ]:
# Cell 2: READ-ONLY DIAGNOSTIC.
# Calls Bybit V5 get-account-info and friends via the bot's
# bybit_client_for(...) helper, running inside the bot's venv with the
# bot's `.env` sourced first (the cell-4 fix from debug_vwap_bybit2.ipynb \u2014
# python3 -c does NOT inherit systemd's EnvironmentFile over SSH).
#
# Prints, in order:
#   1. marginMode + spotMarginMode  (the gate fields T1 keys off)
#   2. The live BTC borrow tier (the max_borrow_btc default to paste in
#      T1's accounts.yaml margin: block)
#   3. Free USDT + free BTC on bybit_2 UNIFIED  (collateral capacity)
#   4. Whether bybit_2 has any open spot-margin borrow positions today

vm_payload = 'import json, os, sys\nsys.path.insert(0, ".")\nfrom src.units.accounts.clients import bybit_client_for\n\ncfg = {\n    "account_id": "bybit_2",\n    "exchange": "bybit",\n    "api_key_env": "BYBIT_API_KEY_2",\n    "market_type": "spot",\n}\n\nclient = bybit_client_for(cfg)\nif client is None:\n    print("client=None \\u2014 BYBIT_API_KEY_2 / BYBIT_API_SECRET_2 missing in .env")\n    sys.exit(0)\n\ndef _try(label, fn):\n    try:\n        out = fn()\n        print(f"--- {label} ---")\n        print(json.dumps(out, indent=2, default=str)[:4000])\n        print()\n        return out\n    except Exception as exc:\n        print(f"--- {label} FAILED ---")\n        print(f"{type(exc).__name__}: {exc}")\n        print()\n        return None\n\n# 1. Account info — marginMode + spotMarginMode are the gate fields.\nacct = _try("get_account_info", lambda: client.get_account_info())\n\n# Surface the two fields explicitly so the operator does not have to scan JSON.\nmargin_mode = None\nspot_margin_status = None\nunified_status = None\nif isinstance(acct, dict):\n    res = acct.get("result", {}) if isinstance(acct.get("result"), dict) else {}\n    margin_mode = res.get("marginMode")\n    spot_margin_status = (\n        res.get("spotMarginMode")\n        or res.get("spotMarginStatus")\n        or res.get("isSpotMarginTrade")\n    )\n    unified_status = res.get("unifiedMarginStatus")\n\nprint("=" * 60)\nprint("GATE FIELDS (the two values T1 keys off):")\nprint(f"  marginMode             = {margin_mode!r}   (need: REGULAR_MARGIN)")\nprint(f"  spotMarginMode/status  = {spot_margin_status!r}   (need: enabled / 1 / True)")\nprint(f"  unifiedMarginStatus    = {unified_status!r}")\nprint("=" * 60)\nprint()\n\nspot_margin_enabled = str(spot_margin_status).lower() in (\n    "1", "true", "regular", "regular_margin", "on", "enabled",\n)\nprint(f"\\u2192 spotMarginEnabled (computed): {spot_margin_enabled}")\nprint()\n\n# 2. Borrow tier for BTC. Try every pybit method name we know about; any\n# one of them returning a number is enough for the operator to capture\n# the max_borrow_btc default for accounts.yaml.\ndef _btc_borrow_via_spot_margin_data():\n    fn = getattr(client, "get_spot_margin_data", None)\n    if fn is None:\n        raise AttributeError("HTTP.get_spot_margin_data not present on this pybit version")\n    return fn(coin="BTC")\n\ndef _btc_borrow_via_collateral_info():\n    fn = getattr(client, "get_collateral_info", None)\n    if fn is None:\n        raise AttributeError("HTTP.get_collateral_info not present on this pybit version")\n    return fn(currency="BTC")\n\ndef _btc_borrow_via_borrow_quota():\n    fn = getattr(client, "get_borrow_quota", None)\n    if fn is None:\n        raise AttributeError("HTTP.get_borrow_quota not present on this pybit version")\n    return fn(category="spot", symbol="BTCUSDT", side="Sell")\n\n# Per Bybit V5 docs the canonical path for UTA Spot-Margin tier data is\n# /v5/spot-margin-trade/data; collateral-info also returns max-borrow.\n_try("spot_margin_data (UTA per-coin tier)", _btc_borrow_via_spot_margin_data)\n_try("collateral_info BTC", _btc_borrow_via_collateral_info)\n_try("borrow_quota spot BTCUSDT Sell", _btc_borrow_via_borrow_quota)\n\n# 3. Wallet free USDT + free BTC — collateral capacity.\ndef _wallet():\n    return client.get_wallet_balance(accountType="UNIFIED")\n\nwallet = _try("wallet UNIFIED", _wallet)\nif isinstance(wallet, dict):\n    accs = wallet.get("result", {}).get("list", []) or []\n    coins = (accs[0].get("coin") if accs else []) or []\n    keep = ["USDT", "BTC"]\n    rows = [c for c in coins if c.get("coin") in keep]\n    print("=" * 60)\n    print("FREE BALANCES on bybit_2 (UNIFIED):")\n    for r in rows:\n        coin = r.get("coin")\n        free = r.get("availableToWithdraw") or r.get("free") or r.get("walletBalance")\n        borrow = r.get("borrowAmount") or r.get("totalOrderIM")\n        print(f"  {coin:5s}  free={free!s:18s}  borrow={borrow!s}")\n    if not rows:\n        print("  (no USDT or BTC rows in result)")\n    print("=" * 60)\n    print()\n\n# 4. Open spot-margin BORROW positions on bybit_2 today.\ndef _borrows():\n    fn = getattr(client, "get_borrow_history", None)\n    if fn is None:\n        raise AttributeError("HTTP.get_borrow_history not present on this pybit version")\n    return fn(currency="BTC", limit=20)\n\nborrows = _try("borrow_history BTC (last 20)", _borrows)\nprint("=" * 60)\nprint("OPEN BORROW POSITIONS (informational \\u2014 should be empty pre-T1):")\nif isinstance(borrows, dict):\n    rows = borrows.get("result", {}).get("list", []) or []\n    if not rows:\n        print("  (no borrow history rows \\u2014 OK, fresh slate)")\n    else:\n        # Bybit returns historical borrows; we only flag rows whose status\n        # implies an open / unpaid borrow. If the operator sees any here,\n        # that needs investigation BEFORE T1.\n        for r in rows[:10]:\n            status = r.get("status") or r.get("type")\n            print(f"  ts={r.get(\'createdTime\')} coin={r.get(\'currency\')} qty={r.get(\'borrowAmount\')} status={status}")\nelse:\n    print("  (borrow history call did not return JSON \\u2014 see error above)")\nprint("=" * 60)\n'

# Stage the payload on the VM (writing via stdin keeps secrets and
# whitespace intact \u2014 no shell-escape minefield).
stage_proc = subprocess.run(
    ["ssh"] + ssh_opts + [ssh_target,
        "cat > /tmp/s047_t0_check.py && chmod 600 /tmp/s047_t0_check.py"],
    input=vm_payload, text=True, capture_output=True,
)
if stage_proc.returncode != 0:
    raise SystemExit(
        f"\u274c staging payload failed: {stage_proc.stderr[:400]}"
    )

# Run with .env sourced (mirrors the systemd EnvironmentFile environment).
output = run_ssh(
    f"cd {REPO_DIR} && set -a && . ./.env && set +a && "
    f"/usr/bin/python3 /tmp/s047_t0_check.py 2>&1; rm -f /tmp/s047_t0_check.py",
    label="bybit_2 spot-margin probe", allow_fail=True,
)
print(output or "(no output \u2014 something failed silently; re-check Cell 1)")


---

## What to do based on Cell 2's output

Read top to bottom; the **first** rule that matches is the one to act on.

**Reminder on rules:** the dispatcher's `live | dry_run` switch is the
only canonical execution gate in the system (`docs/claude/workplan.md` §
"Live / dry-run rule"). This notebook does **not** add a gate — it is a
read-only data-collection step that informs the parameters T2's
RiskManager work will use. None of the rows below imply "do not trade
on bybit_2" — they imply "the parameters the risk manager will receive
are only meaningful once these values are real."

| What Cell 2 printed | What it means | Next step |
|---|---|---|
| `client=None — BYBIT_API_KEY_2 / BYBIT_API_SECRET_2 missing in .env` | bybit_2 has no exchange creds on this VM | Run `notebooks/operator/rotate_api_keys.ipynb` first, then re-run Cell 2. |
| `marginMode=None` and every `_try(...)` block printed `FAILED` | The signed call failed at the auth layer (key revoked, IP-locked, or wrong scope) | Check the API key still exists on Bybit; re-run `rotate_api_keys.ipynb` if it was rotated. |
| `marginMode = 'REGULAR_MARGIN'` **and** `spotMarginEnabled (computed) = True` **and** the BTC borrow tier printed a number | ✅ Spot Margin is on; the BTC max-borrow tier number is what T2's risk-manager rules will treat as the per-account `max_borrow_btc` parameter. Capture that number. | Note the BTC max-borrow value somewhere you can paste into the T1 PR comment thread. Re-read **Cell 4**. |
| `marginMode = 'REGULAR_MARGIN'` **but** `spotMarginEnabled (computed) = False` | Account is on Regular Margin but the **Spot** subset of margin is still disabled. Until the operator flips it, Bybit will reject every spot order with `isLeverage=1` server-side (retCode 110007). T1+T2 code can still ship; it just won't actually execute on bybit_2 until the toggle is on. | **Operator action (informational, not gating):** Bybit web UI → Account → Margin Mode → Enable Spot Margin. Then re-run Cell 2 to capture the borrow tier. |
| `marginMode = 'ISOLATED_MARGIN'` or `'PORTFOLIO_MARGIN'` | The account is on a non-Regular margin mode that does not expose Spot Margin via `isLeverage=1`. Same situation as the row above — Bybit will reject server-side until the operator switches modes. | **Operator action:** switch to Regular Margin in the Bybit web UI, enable Spot Margin, re-run Cell 2. |
| `OPEN BORROW POSITIONS` lists rows with non-zero `qty` and a non-`Repaid` status | bybit_2 already has open spot-margin borrows. The risk manager will need to know about them when sizing (T2 work) so they don't get double-counted as available collateral. | Note the open-borrow rows for the T2 PR comment thread so the risk-manager rules account for them. |
| BTC borrow tier shows `0` or `None` everywhere | Spot Margin is enabled but the account has no BTC borrow allowance on this tier. The risk-manager `max_borrow_btc` parameter would default to 0 — meaningful, but the trader will only ever buy on bybit_2 in that state, never short. | If you want bybit_2 to be able to short, add USDT collateral / move up a VIP tier on Bybit, then re-run Cell 2. Otherwise just record `max_borrow_btc=0` for the T1 PR. |

After acting on whichever row matches, **re-run Cell 2**. The check is
idempotent.


---

## Hand-off to S-047 T1 / T2

This notebook collects the live parameter values the next checkpoints
will use. T1 (config plumbing) and T2 (risk-manager rules) are *not*
blocked by anything in this notebook — they can ship at any time. The
notebook just makes sure the numbers they're plumbing through and
deciding off of are real, not made up.

What to capture from Cell 2 for the T1 / T2 PR comment thread:

- The BTC max-borrow tier number (any of the three `_try(...)` blocks
  that returned a value works — they all read the same Bybit
  endpoint family).
- bybit_2's free USDT (collateral capacity).
- Any open borrow positions, with their currency + amount, so T2's
  risk-manager rules can subtract them from available collateral
  rather than treating the whole free balance as usable.

Compliance note: nothing in this notebook adds a runtime gate. Per
`docs/claude/workplan.md` § "Live / dry-run rule" the dispatcher
maintains the **only** canonical execution gate; spot-margin support
ships as routing changes (always send `isLeverage=1` on bybit_2) plus
risk-manager rules (sizing from USDT collateral, accounting for
borrow fees and liquidation distance), never as a new refuse-to-trade
branch outside the risk manager.

### Reference

- Sprint plan: `docs/sprint-plans/S-047-bybit2-spot-margin.md`.
- Diagnostic recap (S-047 trigger session): `notebooks/operator/debug_vwap_bybit2.ipynb`.
- One-canonical-gate rule: `docs/claude/workplan.md` § "Live / dry-run rule".
- Operator-notebook contract: `docs/claude/colab-workflows.md` § "Operator VM steps" + Rule 7.
